## Testes automatizados

Foram implementados testes unitários utilizando Pytest para validar os principais componentes da solução:

- Validações de qualidade de dados;
- Regras de negócio;
- Operações de MERGE;
- Implementação SCD Tipo 2.

Execução realizada através do notebook `99_orquestrador_testes`.

Resultado da execução:

- ✅ 7 testes executados
- ✅ 7 testes aprovados
- ✅ 0 falhas

In [0]:
import os
import sys
import shutil
import importlib
import contextlib
import io
from pathlib import Path

import pytest


ORIGEM = Path(
    "/Workspace/Users/luizaautran@gmail.com/desafio_sincred"
)
DESTINO = Path("/tmp/desafio_sincred")
PASTA_TESTES = DESTINO / "testes" / "unitarios"

# Limpa testes carregados anteriormente
for nome_modulo in list(sys.modules):
    if (
        nome_modulo.startswith("test_")
        or nome_modulo.startswith("testes.")
    ):
        sys.modules.pop(nome_modulo, None)

importlib.invalidate_caches()

# Copia novamente o projeto atualizado
shutil.rmtree(DESTINO, ignore_errors=True)
shutil.copytree(ORIGEM, DESTINO)

print(f"Origem existe: {ORIGEM.exists()}")
print(f"Destino existe: {DESTINO.exists()}")
print(f"Pasta de testes existe: {PASTA_TESTES.exists()}")

print("\nArquivos encontrados:")

for arquivo in sorted(PASTA_TESTES.glob("test_*.py")):
    print(f"- {arquivo.name}")

assert PASTA_TESTES.exists(), (
    f"Pasta não encontrada: {PASTA_TESTES}"
)

assert len(list(PASTA_TESTES.glob("test_*.py"))) == 4, (
    "Não foram encontrados os quatro arquivos de teste."
)

os.chdir(DESTINO)

print(f"\nDiretório atual: {os.getcwd()}")

In [0]:
import contextlib
import io
import pytest

arquivo_junit = DESTINO / "resultado_testes.xml"
arquivo_log = DESTINO / "resultado_testes.txt"

saida = io.StringIO()

with contextlib.redirect_stdout(saida), contextlib.redirect_stderr(saida):
    codigo_saida = pytest.main(
        [
            str(PASTA_TESTES),
            "-v",
            "--tb=short",
            "-p",
            "no:cacheprovider",
            f"--rootdir={DESTINO}",
            f"--junitxml={arquivo_junit}",
        ]
    )

conteudo_log = saida.getvalue()
arquivo_log.write_text(conteudo_log, encoding="utf-8")

print(conteudo_log)
print("=" * 70)
print(f"Código de saída: {codigo_saida}")
print(f"Log: {arquivo_log}")
print(f"Relatório JUnit: {arquivo_junit}")

if codigo_saida == 0:
    print("RESULTADO FINAL: TODOS OS TESTES PASSARAM")
else:
    print("RESULTADO FINAL: EXISTEM TESTES COM FALHA")